In [1]:
import copy

import numpy

import cloudvolume
import kimimaro
import fastremap

np = numpy

In [15]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

from ac_pcg.pcgraph.edges import Edges
from ac_pcg.pcgraph.edges import EDGE_TYPES
from ac_pcg.io.edges import put_chunk_edges

In [3]:
import gzip
import pathlib
import pickle

def read_gzip_array(fn, preprocess_func=lambda x: x):
    with gzip.open(fn, "rb") as f:
        a = numpy.load(f)
    return preprocess_func(a)

test_data_path = pathlib.Path(
    "/allen/programs/celltypes/workgroups/em-connectomics/russelt/pcg_axconn/test_data_strip/"
)

test_data_labeled_array_path = test_data_path / "H17_x55_S32_230412_Pos42.npy.gz"
test_skels_path = test_data_path / "H17_x55_S32_230412_Pos42.skels.pkl"

In [4]:
labeled_array = read_gzip_array(test_data_labeled_array_path)
with test_skels_path.open(mode="rb") as skels_fobj:
    label_skels = pickle.load(skels_fobj)

In [5]:
%%time
import rtree

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}


p = rtree.index.Property()
p.dimension = 3
label_skel_idx = rtree.index.Index(
    ((sk_id, skel_bb.to_list(), label_skels[sk_id]) for sk_id, skel_bb in skel_id_to_bboxes.items()),
    properties=p
)

CPU times: user 4.52 s, sys: 68.5 ms, total: 4.59 s
Wall time: 4.62 s


In [6]:
labeled_array_bbox = cloudvolume.CloudVolume
labeled_array_chunksize = numpy.array((128, 128, 128))

In [7]:
%time skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}

CPU times: user 1.68 s, sys: 132 μs, total: 1.68 s
Wall time: 1.68 s


In [8]:
def get_bbox_chunks(bbox, chunk_size, offset=None):
    if offset is not None:
        raise NotImplementedError
    (chunk_min, chunk_max), (remainder_min, remainder_max) = numpy.divmod(
        numpy.array([bbox.minpt, bbox.maxpt]), chunk_size)
    # chunk_max += remainder_max.astype(bool)

    # TODO could be more clever about dims
    imin, jmin, kmin = chunk_min
    imax, jmax, kmax = chunk_max
    chunks = np.mgrid[imin:imax+1:1, jmin:jmax+1:1, kmin:kmax+1:1].reshape(3, -1).T

    return chunks

In [9]:
%time skel_to_chunks = {sk_id: get_bbox_chunks(sk_bbox, labeled_array_chunksize) for sk_id, sk_bbox in skel_id_to_bboxes.items()}

CPU times: user 2.4 s, sys: 48 μs, total: 2.4 s
Wall time: 2.39 s


In [10]:
%time
chunk_to_skel_ids = {}
for sk_id, sk_chunks in skel_to_chunks.items():
    for sk_chunk in sk_chunks:
        try:
            chunk_to_skel_ids[tuple(sk_chunk)].append(sk_id)
        except KeyError:
            chunk_to_skel_ids[tuple(sk_chunk)] = [sk_id]

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 10 μs


In [11]:
# TODO multiprocess futures w/ skeleton processing

In [12]:
def chunk_idx_to_bbox(chunk_idx, chunk_size, chunked_box_shape):
    chunk_mins = tuple(idx * chunk_d for idx, chunk_d in zip(chunk_idx, chunk_size))
    chunk_max = tuple(min(box_d, chunk_min + chunk_d) for chunk_min, chunk_d, box_d in zip(chunk_mins, chunk_size, chunked_box_shape))
    bbox = cloudvolume.Bbox(chunk_mins, chunk_max)
    return ac_pcg.chunks.ChunkBbox(
        bbox=bbox,
        chunk_idx=chunk_idx
    )

In [13]:
def process_oversegment_array(arr, skels, label_func, oversegment_kwargs=None):
    oversegment_kwargs = oversegment_kwargs or {}

    oversegmented_arr, oversegmented_skels = kimimaro.utility.oversegment(
        arr, skels, **oversegment_kwargs)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: label_func(input_lbl)  # input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_arr
        )
    }
    relabeled_arr = fastremap.remap(oversegmented_arr, lbl_map)

    for skel in oversegmented_skels:
        skel.segments = fastremap.remap(skel.segments, lbl_map)

    return relabeled_arr, oversegmented_skels

In [14]:
import time

output_skels = copy.deepcopy(label_skels)
labeler = ac_pcg.label.ChunkLabeler()
output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

tic = time.time()

for chunk_num, (chunk_idx, chunk_skel_ids) in enumerate(chunk_to_skel_ids.items()):
    chunk_box = chunk_idx_to_bbox(chunk_idx, labeled_array_chunksize, labeled_array.shape)
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(label_skels[chunk_skel_id], chunk_contains_bb)
        for chunk_skel_id in chunk_skel_ids))

    try:
        skels, skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]

    relabeled_arr, relabeled_skels = process_oversegment_array(
        subvol_arr, skels,
        label_func=lambda x: labeler.encode_chunk_seg(chunk_idx, x),
        oversegment_kwargs={
            "downsample": 6,
            "progress": False
        })

    output_arr[chunk_box.bbox.to_slices()] = relabeled_arr[...]
    
    for i, (skel, subvol_indices) in enumerate(zip(relabeled_skels, skel_indices)):
        output_skel = output_skels[skel.id]
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments
    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

print(time.time() - tic)

0 0.5251107215881348
10 4.941328048706055
20 8.80139970779419
30 12.34648060798645
40 15.298853158950806
50 18.278961181640625
60 21.593491315841675
70 25.27192258834839
80 28.02658700942993
90 31.24798560142517
100 35.987223625183105
110 40.46453857421875
120 43.24388885498047
130 46.411824226379395
140 49.8888144493103
150 52.72693920135498
160 55.60246968269348
170 58.33951997756958
180 60.94796299934387
190 64.68743968009949
200 67.8853406906128
210 69.91235446929932
220 72.23836421966553
230 73.87558388710022
240 75.1389365196228
250 76.01630711555481
260 77.87627744674683
270 81.44285893440247
280 83.79941582679749
290 86.48633980751038
300 89.08037853240967
310 90.1851601600647
320 93.96270394325256
330 97.55637168884277
340 100.79829359054565
350 102.87179183959961
360 105.7005352973938
370 108.15682244300842
380 109.21600008010864
390 111.12242579460144
410 112.78596782684326
420 114.25978684425354
430 116.38999915122986
440 118.73327684402466
450 121.23814558982849
460 124.92

In [6]:
# TODO concurrent processing, serial vertex assignment.
#   serial chunk processing method below

In [44]:
# alternatively, run over all chunks

import time

chunk_size = (128, 128, 128)
chunk_boxes = ac_pcg.chunks.iterate_chunk_slice_boxes(
    labeled_array.shape, chunk_size)

labeler = ac_pcg.label.ChunkLabeler()

output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(res.object, chunk_contains_bb)
        for res in label_skel_idx.intersection(
            chunk_contains_bb.to_list(), objects=True
        )
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    oversegmented_subvol_arr, oversegmented_subvol_skels = kimimaro.utility.oversegment(
        subvol_arr, subvol_skels, downsample=6, progress=False)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_subvol_arr
        )
    }
    relabeled_arr = fastremap.remap(oversegmented_subvol_arr, lbl_map)
    
    output_arr[chunk_box.bbox.to_slices()] = relabeled_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        skel.segments = fastremap.remap(skel.segments, lbl_map)
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments
    

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

print(time.time() - tic)

90 0.32837820053100586
100 1.028796672821045
120 3.537492513656616
130 5.14408278465271
140 6.592854261398315
150 8.169435501098633
160 9.642114877700806
170 11.172497272491455
180 13.025489091873169
190 15.021819829940796
200 16.633267641067505
210 18.368619680404663
220 20.40191674232483
230 22.222224712371826
240 24.074891567230225
250 26.04107689857483
260 27.84222149848938
270 30.310452222824097
280 33.09475088119507
290 35.661949157714844
300 38.37808418273926
310 41.02535939216614
320 42.99960494041443
330 44.8966178894043
340 46.56064701080322
350 48.7562255859375
360 50.96120262145996
370 53.29250955581665
390 57.11000084877014
400 59.51981472969055
420 63.529720306396484
430 65.45258665084839
450 69.54356861114502
460 71.70915126800537
480 75.42686033248901
490 77.20927953720093
500 78.87331461906433
510 80.54965591430664
520 82.31556606292725
530 83.87360334396362
540 85.7892198562622
550 87.5920901298523
560 89.04762053489685
570 90.49724817276001
580 92.06546235084534
590 

In [8]:
# write out protobuf edges and precomputed array

In [16]:
def label_chunk(labeler, chunk):
    layer_offset = numpy.uint64(64 - labeler.n_bits_for_layer_id)
    x_offset = numpy.uint64(layer_offset - labeler.spatial_bit_count)
    y_offset = numpy.uint64(x_offset - labeler.spatial_bit_count)
    z_offset = numpy.uint64(y_offset - labeler.spatial_bit_count)

    layer = numpy.uint64(labeler.level)
    x, y, z = numpy.uint64(chunk)

    segid_bits = numpy.uint64(64 - labeler.n_bits_for_layer_id - 3 * labeler.spatial_bit_count)

    return numpy.uint64(
      layer << layer_offset | x << x_offset | y << y_offset | z << z_offset
    ) >> segid_bits


def chunk_edges_from_skeleton(skel, query_chunk, labeler):
    seg_edges = skel.segments[skel.edges]
    reduced_seg_edges = numpy.unique(
        numpy.sort(
            seg_edges[
            (seg_edges[:, 0] != seg_edges[:, 1])
            ], axis=1
        ),
        axis=0)

    segid_bits = numpy.uint64(64 - labeler.n_bits_for_layer_id - 3 * labeler.spatial_bit_count)                                                                                                                                                                                                       

    # calculate layer and chunk label for edges
    layer_chunk_edges = reduced_seg_edges >> segid_bits

    # calculate chunk label and find occurences
    query_layer_chunk_idx = label_chunk(labeler, query_chunk)
    query_chunk_edges_mask = (layer_chunk_edges == query_layer_chunk_idx)

    in_chunk_edges_mask = query_chunk_edges_mask[:, 0] & query_chunk_edges_mask[:, 1]
    between_chunk_edges_mask = query_chunk_edges_mask[:, 0] ^ query_chunk_edges_mask[:, 1]    

    in_chunk_edges = reduced_seg_edges[in_chunk_edges_mask]
    between_chunk_edges = reduced_seg_edges[between_chunk_edges_mask]
    return in_chunk_edges, between_chunk_edges

In [17]:
edge_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_pcg_example/H17_x55_S32_230412_Pos42/edges"
for chunk, chunk_skel_ids in chunk_to_skel_ids.items():
    in_chunk_edge_results = []
    between_chunk_edge_results = []
    for skel_id in chunk_skel_ids:
        # TODO probably just use edges of all skels r.t. iterate over skel vertices
        skel = output_skels[skel_id]
        skel_in_chunk_edge_arr, skel_between_chunk_edge_arr = chunk_edges_from_skeleton(skel, chunk, labeler)
        in_chunk_edge_results.append(skel_in_chunk_edge_arr)
        between_chunk_edge_results.append(skel_between_chunk_edge_arr)

    in_chunk_edge_arr = numpy.concatenate(in_chunk_edge_results)
    between_chunk_edge_arr = numpy.concatenate(between_chunk_edge_results)
    in_chunk_edges = Edges(*in_chunk_edge_arr.T)
    between_chunk_edges = Edges(*between_chunk_edge_arr.T)
    # cross_chunk_edges are not generated in this segmentation method
    cross_chunk_edges = Edges([], [])

    edges_d = {
        EDGE_TYPES.in_chunk: in_chunk_edges,
        EDGE_TYPES.between_chunk: between_chunk_edges,
        EDGE_TYPES.cross_chunk: cross_chunk_edges
    }
    
    put_chunk_edges(edge_output_loc, chunk, edges_d, compression_level=22)

In [ ]:
# write out labeled supervoxel volume

In [21]:
list(labeled_array.shape)

[21568, 288, 288]

In [22]:
label_output_loc = "file:///allen/programs/celltypes/workgroups/em-connectomics/russelt/ac_pcg_example/H17_x55_S32_230412_Pos42/labels"


In [26]:
cv_info = {
    "data_type": "uint64",
    "num_channels": 1,
    "scales": [
        {
            "chunk_sizes": [
                [
                    128,
                    128,
                    128
                ]
            ],
            "compressed_segmentation_block_size": (8, 8, 8),
            "encoding": "compressed_segmentation",
            "key": "1000_1000_1000",
            "resolution": [
                1000,
                1000,
                1000
            ],
            "size": output_arr.shape,
            "voxel_offset": [
                0,
                0,
                0
            ]
        }
    ],
    "type": "segmentation"
}

In [27]:
seg_cv = cloudvolume.CloudVolume(label_output_loc, info=cv_info)

In [28]:
seg_cv.commit_info()

In [31]:
seg_cv[..., 0] = output_arr[...]

Uploading: 100%|████████████████████████████████████████████████████████████████████| 1521/1521 [01:04<00:00, 23.49it/s]
